In [3]:
##### Run raster model using country average intensities 

import pandas as pd
from pathlib import Path
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
import joblib
import rasterio
import gc

In [4]:
##### SET-UP

### Set directories 
# Get the current working directory
cd = Path.cwd().parent.parent 

### Import data 

# Production data (raster)
production_path = f'{cd}/Data/Clean/Production/total_production_tonnes_2020.tif'

# Capital data (raster)
capital_int_path = f'{cd}/Data/Clean/Predictors/Rasters/country_capital_intensity_tonnes.tif'

# Labor data (raster)
labor_int_path = f'{cd}/Data/Clean/Predictors/Rasters/country_labor_intensity_tonnes.tif'

# Set paths 
capital_path = f"{cd}/Results/Raster_model/country_avg_model/capital_USD.tif"
labor_path = f"{cd}/Results/Raster_model/country_avg_model/jobs.tif"

In [5]:
##### CONVERT INTENSITIES TO CAPITAL AND LABOR

def multiply_rasters_and_save(raster1_path, raster2_path, out_path):
    with rasterio.open(raster1_path) as src1, rasterio.open(raster2_path) as src2:
        arr1 = src1.read(1).astype('float32')
        arr2 = src2.read(1).astype('float32')
        profile = src1.profile.copy()

        # handle nodata so it doesn't get multiplied into garbage
        nodata1 = src1.nodata
        nodata2 = src2.nodata
        mask = np.ones(arr1.shape, dtype=bool)
        if nodata1 is not None:
            mask &= (arr1 != nodata1)
        if nodata2 is not None:
            mask &= (arr2 != nodata2)

        out_nodata = -9999.0
        result = np.full(arr1.shape, out_nodata, dtype='float32')
        result[mask] = arr1[mask] * arr2[mask]

        profile.update(dtype='float32', nodata=out_nodata, count=1)

        with rasterio.open(out_path, 'w', **profile) as dst:
            dst.write(result, 1)

# Capital
multiply_rasters_and_save(capital_int_path, production_path, capital_path)

# Labor
multiply_rasters_and_save(labor_int_path, production_path, labor_path)
